In [1]:
%load_ext autoreload
%autoreload 2

import warnings

warnings.filterwarnings("ignore")

# Saving & Loading

A fitted `GlassBoxUMAP` is a PyTorch model, plus the bits of state that connect raw input features to that model (PCA basis, feature means, training hyperparameters). The `save` and `load` methods round-trip all of this to a single file on disk.

## Fit a model

We'll use scikit-learn's digits dataset, which is 1,797 handwritten digit images flattened to 64 features (8x8 grayscale pixels).

In [7]:
from glass_box_umap import GlassBoxUMAP

def load_data():
    from sklearn.datasets import load_digits
    from sklearn.preprocessing import StandardScaler
    digits, _ = load_digits(return_X_y=True)
    return StandardScaler().fit_transform(digits)

X = load_data()
embedder = GlassBoxUMAP(quiet=True)
embedder.fit(X)

GlassBoxUMAP(n_neighbors=15, min_dist=0.1, metric='euclidean', n_components=2, random_state=None, encoder_name='glassbox_default_encoder', encoder_kwargs={}, pca_components=None, lr=0.001, epochs=100, batch_size=10000, num_batches=None, negative_sample_rate=5, repulsion_strength=1.0, num_workers=0, checkpoint_dir=None, restore_best_weights=True, quiet=True, extra_callbacks=[], _model=UMAPLightningModule(
  (encoder): DeepPReLUNet(
    (flatten): Flatten(start_dim=1, end_dim=-1)
    (model): Sequential(
      (0): Linear(in_features=64, out_features=128, bias=False)
      (1): VmapPReLU(num_parameters=1)
      (2): LayerNormDetached()
      (3): Dropout(p=0.0, inplace=False)
      (4): Linear(in_features=128, out_features=128, bias=False)
      (5): VmapPReLU(num_parameters=1)
      (6): LayerNormDetached()
      (7): Dropout(p=0.0, inplace=False)
      (8): Linear(in_features=128, out_features=128, bias=False)
      (9): VmapPReLU(num_parameters=1)
      (10): Dropout(p=0.0, inplace=Fa

## Save the model

`save` takes a path and writes a PyTorch checkpoint.

:::{admonition} save API
:class: api, dropdown

From the {meth}`API docs <glass_box_umap.GlassBoxUMAP.save>`:

```{eval-rst}
.. automethod:: glass_box_umap.GlassBoxUMAP.save
    :noindex:
```
:::

In [8]:
from pathlib import Path

model_path = Path.cwd() / "embedder.pt"
embedder.save(model_path)

print(f"saved model ({model_path.stat().st_size / 1024:.1f} KiB) to {model_path.name}")

saved model (167.2 KiB) to embedder.pt


## Load the model

`GlassBoxUMAP.load` is a classmethod that reconstructs the embedder from a checkpoint.


:::{admonition} load API
:class: api, dropdown

From the {meth}`API docs <glass_box_umap.GlassBoxUMAP.load>`:

```{eval-rst}
.. automethod:: glass_box_umap.GlassBoxUMAP.load
    :noindex:
```
:::

In [9]:
loaded = GlassBoxUMAP.load(model_path)

The reloaded embedder is functionally identical to the original. We can confirm that with `==`:

In [10]:
loaded == embedder

True

```{note}
`==` performs a *semantic* comparison: it checks that both embedders describe the same trained model — matching architecture, hyperparameters, learned weights, PCA fit, and centering vector — and ignores incidental runtime state like device placement, logging verbosity, and DataLoader workers. So two consecutive `load`s of the same file always equate, even if one was moved to CPU and the other left on GPU.
```

Concretely, that means `transform` and `compute_contributions` produce bitwise-identical outputs from either embedder:

In [11]:
import numpy as np

Z_original = embedder.transform(X)
Z_loaded = loaded.transform(X)
assert np.array_equal(Z_original, Z_loaded)

C_original = embedder.compute_contributions(X)
C_loaded = loaded.compute_contributions(X)
assert np.array_equal(C_original, C_loaded)

print("embeddings and contributions match exactly")

embeddings and contributions match exactly


```{caution}
Checkpoints are PyTorch [`torch.save`](https://docs.pytorch.org/docs/stable/generated/torch.save.html) files, which use Python's `pickle` under the hood. Only load checkpoints from sources you trust — a malicious checkpoint can execute arbitrary code on load.
```